In [ ]:
# ============================================================
# CELL 1: SETUP — HRVRL integration
# Drive mount, HRVRL official repo clone, pretrained weight download
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# HRVRL official repo — models_rip.py-র exact architecture দরকার,
# নিজে transcribe করলে ভুল হওয়ার ঝুঁকি, তাই official code ব্যবহার করব
!git clone --depth 1 https://github.com/sulab-wmu/HRVRL.git /content/HRVRL

import sys
sys.path.append('/content/HRVRL/finetune')

# Google Drive folder থেকে pretrained weight download-এর জন্য gdown
!pip install -q gdown

import torch, torchvision
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ torchvision: {torchvision.__version__}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# --- pretrained weight download (Google Drive folder) ---
# folder link: https://drive.google.com/drive/folders/1Hzxv36qyyqTgyE0jGICJ-1hlbg0DnSFh
import os
os.makedirs('/content/hrvrl_weights', exist_ok=True)
!gdown --folder "https://drive.google.com/drive/folders/1Hzxv36qyyqTgyE0jGICJ-1hlbg0DnSFh" -O /content/hrvrl_weights

print("\n--- Downloaded files ---")
for root, dirs, files in os.walk('/content/hrvrl_weights'):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {path}  ({size_mb:.1f} MB)")

# --- Data check (GAVE2 dataset) ---
DATA_ROOT = "/content/drive/MyDrive/GAVE2_preliminary"
print(f"\n--- Data check ---")
for sub in ['training/images', 'training/FFA_A', 'training/FFA_AV', 'training/av', 'training/masks']:
    path = f"{DATA_ROOT}/{sub}"
    print(f"  {'✅' if os.path.exists(path) else '❌'} {sub}: "
          f"{len(os.listdir(path)) if os.path.exists(path) else 0}টা file")

print("\n🎯 Cell 1 সম্পন্ন। Cell 2 run করো।")

Mounted at /content/drive
Cloning into '/content/HRVRL'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 165 (delta 8), reused 158 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 51.02 MiB | 19.97 MiB/s, done.
Resolving deltas: 100% (8/8), done.
✅ PyTorch: 2.11.0+cu128
✅ torchvision: 0.26.0+cu128
✅ GPU: Tesla T4
Retrieving folder contents
Processing file 1_78kbhkQlnYUvackqnCt0efnFVtXbGOc G_pretrain.pkl
Processing file 19iEDisqoKPuFTf9ksmrZ5n-JMHudM_z-PM0fYg5wOLg README
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1_78kbhkQlnYUvackqnCt0efnFVtXbGOc
From (redirected): https://drive.google.com/uc?id=1_78kbhkQlnYUvackqnCt0efnFVtXbGOc&confirm=t&uuid=8596097b-0394-4955-ad53-43f27665a064
To: /content/hrvrl_weights/G_pretrain.pkl
100% 116M/116M [00

In [ ]:
# ============================================================
# CELL 2: CHECKPOINT DIAGNOSTIC — architecture বানানোর আগে গঠন যাচাই
# ============================================================

import torch

ckpt_path = '/content/hrvrl_weights/G_pretrain.pkl'
checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)

print(f"Checkpoint টাইপ: {type(checkpoint)}")

if isinstance(checkpoint, dict):
    print(f"\nটপ-লেভেল key: {list(checkpoint.keys())}")

    # models_rip.py-র load_checkpoint অনুযায়ী 'model' বা 'model_ema' খুঁজি
    state_dict = None
    for k in ['model', 'model_ema']:
        if k in checkpoint:
            state_dict = checkpoint[k]
            print(f"\n✅ '{k}' key-তে state_dict পাওয়া গেছে")
            break
    if state_dict is None:
        state_dict = checkpoint
        print(f"\n⚠️ 'model'/'model_ema' key নেই, পুরো dict-ই state_dict ধরছি")
else:
    state_dict = checkpoint
    print("সরাসরি state_dict (dict নয়, অন্য টাইপ)")

print(f"\nমোট parameter key: {len(state_dict)}")
print(f"\n--- প্রথম ১৫টা key ---")
for k in list(state_dict.keys())[:15]:
    v = state_dict[k]
    shape = tuple(v.shape) if hasattr(v, 'shape') else type(v)
    print(f"  {k}  {shape}")

print(f"\n--- শেষ ১০টা key (head-সম্পর্কিত কিনা দেখতে) ---")
for k in list(state_dict.keys())[-10:]:
    v = state_dict[k]
    shape = tuple(v.shape) if hasattr(v, 'shape') else type(v)
    print(f"  {k}  {shape}")

print("\n🎯 Cell 2 সম্পন্ন। Cell 3 run করো।")

Checkpoint টাইপ: <class 'collections.OrderedDict'>

টপ-লেভেল key: ['pg_fusion.gamma_patch_self', 'pg_fusion.gamma_patch_global', 'pg_fusion.patch_query.weight', 'pg_fusion.patch_key.weight', 'pg_fusion.patch_value.weight', 'pg_fusion.patch_global_query.weight', 'pg_fusion.global_key.weight', 'pg_fusion.global_value.weight', 'pg_fusion.fusion.weight', 'pg_fusion.fusion.bias', 'pg_fusion.out_patch.weight', 'pg_fusion.out_patch.bias', 'pg_fusion.out_global.weight', 'pg_fusion.out_global.bias', 'base_layers_global_momentum.0.0.weight', 'base_layers_global_momentum.0.0.bias', 'base_layers_global_momentum.0.1.weight', 'base_layers_global_momentum.0.1.bias', 'base_layers_global_momentum.1.0.layer_scale', 'base_layers_global_momentum.1.0.block.0.weight', 'base_layers_global_momentum.1.0.block.0.bias', 'base_layers_global_momentum.1.0.block.2.weight', 'base_layers_global_momentum.1.0.block.2.bias', 'base_layers_global_momentum.1.0.block.3.weight', 'base_layers_global_momentum.1.0.block.3.bias',

In [ ]:
# ============================================================
# CELL 3: PGNet ARCHITECTURE (HRVRL pretrain/AV/models/network.py থেকে
# হুবহু পুনর্গঠন) + checkpoint load + 5-channel/4-class adaptation
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.convnext import convnext_tiny, ConvNeXt_Tiny_Weights

# --- SaveFeatures: intermediate layer output hook করে skip connection-এর জন্য ---
class SaveFeatures():
    features = None
    def __init__(self, m):
        self.hook = m.register_forward_hook(self.hook_fn)
    def hook_fn(self, module, input, output):
        if len(output.shape) == 3:
            B, L, C = output.shape
            h = int(L ** 0.5)
            output = output.view(B, h, h, C).permute(0, 3, 1, 2).contiguous()
        if len(output.shape) == 4 and output.shape[2] != output.shape[3]:
            output = output.permute(0, 3, 1, 2).contiguous()
        self.features = output
    def remove(self):
        self.hook.remove()


# --- DBlock: decoder upsampling block (skip connection সহ) ---
class DBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels*2, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
        self.conv3 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))

    def forward(self, x, skip):
        if x.shape[1] != skip.shape[1]:
            x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)
        x = self.conv1(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv2(x)
        x = self.conv3(x)
        return x


# --- SegmentationHead: চূড়ান্ত output layer ---
class SegmentationHead(nn.Module):
    def __init__(self, in_channels, num_class, kernel_size=3, upsample=4):
        super().__init__()
        self.upsample = nn.UpsamplingBilinear2d(scale_factor=upsample) if upsample > 1 else nn.Identity()
        self.conv = nn.Conv2d(in_channels, num_class, kernel_size=kernel_size, padding=kernel_size//2)
    def forward(self, x):
        return self.conv(self.upsample(x))


# --- PGNet: পূর্ণ architecture (pretrain/AV/models/network.py-এর সরলীকৃত version) ---
class PGNet(nn.Module):
    def __init__(self, input_ch=3, num_classes=3, pretrained=False):
        super().__init__()
        base_model = convnext_tiny
        cut = 6
        layers = list(base_model(weights=None).features)[:cut]
        base_layers = nn.Sequential(*layers)

        # skip connection hook: stem(96) → stage1(96) → stage2(192) → stage3(384)
        self.stage = []
        self.stage.append(SaveFeatures(base_layers[0][1]))
        self.stage.append(SaveFeatures(base_layers[1][2]))
        self.stage.append(SaveFeatures(base_layers[3][2]))
        self.stage.append(SaveFeatures(base_layers[5][8]))

        self.up2 = DBlock(384, 192)
        self.up3 = DBlock(192, 96)
        self.up4 = DBlock(96, 96)
        self.seg_head = SegmentationHead(96, num_classes, 3, upsample=4)
        self.sn_unet = base_layers

    def forward(self, x):
        x = self.sn_unet(x)
        if len(x.shape) == 4 and x.shape[2] != x.shape[3]:
            x = x.permute(0, 3, 1, 2).contiguous()
        feature = self.stage[::-1]
        skip = feature[1:]
        x = self.up2(x, skip[0].features)
        x = self.up3(x, skip[1].features)
        x = self.up4(x, skip[2].features)
        return self.seg_head(x)

    def close(self):
        for sf in self.stage: sf.remove()


# ============================================================
# checkpoint load: প্রথমে original (3ch input, 3 class output) architecture-এ
# ============================================================
print("PGNet (original: 3-channel input, 3-class output) তৈরি করছি...")
pg_original = PGNet(input_ch=3, num_classes=3)

ckpt_path = '/content/hrvrl_weights/G_pretrain.pkl'
checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
raw_state = checkpoint if 'model' not in checkpoint else checkpoint['model']

model_state = pg_original.state_dict()
matched = {k: v for k, v in raw_state.items()
           if k in model_state and model_state[k].shape == v.shape}
model_state.update(matched)
pg_original.load_state_dict(model_state)
print(f"✅ Original architecture-এ match: {len(matched)}/{len(model_state)}")

# --- forward test (256x256, ছোট ছবিতে দ্রুত যাচাই) ---
pg_original.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, 256, 256)
    out = pg_original(dummy)
print(f"✅ Output shape: {out.shape}  (expected: 1,3,256,256 — full resolution!)")

print("\n🎯 Cell 3 সম্পন্ন। এখন 5-channel/4-class adaptation করব (Cell 4)।")

PGNet (original: 3-channel input, 3-class output) তৈরি করছি...
✅ Original architecture-এ match: 203/203
✅ Output shape: torch.Size([1, 3, 256, 256])  (expected: 1,3,256,256 — full resolution!)

🎯 Cell 3 সম্পন্ন। এখন 5-channel/4-class adaptation করব (Cell 4)।


In [ ]:
# ============================================================
# CELL 4 (fixed): PGNet-এ প্রকৃতপক্ষে 5-channel stem বসানো
# (আগের bug: input_ch parameter নেওয়া হয়েছিল কিন্তু ব্যবহার হয়নি)
# ============================================================

print("PGNet (সঠিকভাবে adapted: 5-channel input, 4-class output) তৈরি করছি...")

# --- ধাপ ১: 3-channel original architecture বানাই (pretrained load-এর জন্য) ---
pg_model = PGNet(input_ch=3, num_classes=4)   # num_classes=4 দিয়েই বানাই, seg_head এমনিতেই নতুন হবে

model_state = pg_model.state_dict()
matched, skipped = {}, []
for k, v in raw_state.items():
    if k in model_state and model_state[k].shape == v.shape:
        matched[k] = v
    elif k in model_state:
        skipped.append((k, tuple(model_state[k].shape), tuple(v.shape)))

model_state.update(matched)
pg_model.load_state_dict(model_state)
print(f"✅ ধাপ ১ — 3-channel অবস্থায় match: {len(matched)}/{len(model_state)}")
for k, new_shape, old_shape in skipped:
    print(f"   বাদ: {k}: pretrained {old_shape} → নতুন {new_shape}")

# --- ধাপ ২: stem conv সরাসরি 5-channel-এ প্রসারিত করি ---
old_conv = pg_model.sn_unet[0][0]   # Conv2d(3, 96, kernel_size=4, stride=4)
old_weight = old_conv.weight.data   # shape (96, 3, 4, 4)

new_conv = nn.Conv2d(5, 96, kernel_size=4, stride=4, bias=(old_conv.bias is not None))
with torch.no_grad():
    new_conv.weight[:, :3, :, :] = old_weight          # প্রথম ৩ channel (CFP RGB) — pretrained
    nn.init.kaiming_normal_(new_conv.weight[:, 3:, :, :])  # শেষ ২ channel (FFA_diff, FFA_A) — নতুন
    if old_conv.bias is not None:
        new_conv.bias[:] = old_conv.bias.data

pg_model.sn_unet[0][0] = new_conv
print(f"\n✅ ধাপ ২ — stem conv বদলানো হলো: Conv2d(3→5, ...) "
      f"প্রথম ৩ channel pretrained, বাকি ২টা নতুন init")

# --- forward test ---
IMG_SIZE = 1024
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pg_model = pg_model.to(DEVICE)
pg_model.train()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
with torch.amp.autocast('cuda'):
    dummy = torch.randn(2, 5, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out = pg_model(dummy)
print(f"\n✅ Output shape: {out.shape}  (expected: 2,4,{IMG_SIZE},{IMG_SIZE})")

peak = torch.cuda.max_memory_allocated()/1e9
print(f"✅ Peak VRAM (forward): {peak:.1f} GB")

total = sum(p.numel() for p in pg_model.parameters())
print(f"✅ Total params: {total/1e6:.1f}M")
print(f"   (তুলনা: EfficientNet-B5 ~30M, HRNet-W32 36.0M, ConvNeXt-DINOv3 53.6M)")

del dummy, out
torch.cuda.empty_cache()
print("\n🎯 Cell 4 সম্পন্ন। Cell 5 run করো।")

PGNet (সঠিকভাবে adapted: 5-channel input, 4-class output) তৈরি করছি...
✅ ধাপ ১ — 3-channel অবস্থায় match: 201/203
   বাদ: seg_head.conv.weight: pretrained (3, 96, 3, 3) → নতুন (4, 96, 3, 3)
   বাদ: seg_head.conv.bias: pretrained (3,) → নতুন (4,)

✅ ধাপ ২ — stem conv বদলানো হলো: Conv2d(3→5, ...) প্রথম ৩ channel pretrained, বাকি ২টা নতুন init

✅ Output shape: torch.Size([2, 4, 1024, 1024])  (expected: 2,4,1024,1024)
✅ Peak VRAM (forward): 4.5 GB
✅ Total params: 14.8M
   (তুলনা: EfficientNet-B5 ~30M, HRNet-W32 36.0M, ConvNeXt-DINOv3 53.6M)

🎯 Cell 4 সম্পন্ন। Cell 5 run করো।


In [ ]:
# ============================================================
# CELL 5: DATASET (আগের EfficientNet/HRNet experiment-এর হুবহু কপি)
# ============================================================

from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import random

DATA_ROOT = "/content/drive/MyDrive/GAVE2_preliminary"
TRAIN_DIR = f"{DATA_ROOT}/training"
SAVE_PATH = f"{DATA_ROOT}/best_model_hrvrl_pgnet.pth"
NUM_CLASSES = 4
BATCH_SIZE = 2
NUM_WORKERS = 2

def parse_av_label(label_path):
    img = np.array(Image.open(label_path).convert("RGB"))
    R, G, B = img[:,:,0], img[:,:,1], img[:,:,2]
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    mask[(R>150) & (G<50)  & (B<50)]  = 1
    mask[(R<50)  & (G<50)  & (B>150)] = 2
    mask[(R<50)  & (G>150) & (B<50)]  = 3
    return mask


class GAVE2Dataset(Dataset):
    def __init__(self, case_ids, augment=False):
        self.case_ids = case_ids
        self.augment  = augment

    def __len__(self):
        return len(self.case_ids)

    def _load_rgb(self, path):
        img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2,0,1)

    def _load_gray(self, path):
        img = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).unsqueeze(0)

    def __getitem__(self, idx):
        case = f"g_{self.case_ids[idx]+1:03d}"
        base = TRAIN_DIR

        cfp    = self._load_rgb(f"{base}/images/{case}.png")
        ffa_a  = self._load_gray(f"{base}/FFA_A/{case}.png")
        ffa_av = self._load_gray(f"{base}/FFA_AV/{case}.png")

        label = parse_av_label(f"{base}/av/{case}.png")
        label_pil = Image.fromarray(label).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
        label = torch.from_numpy(np.array(label_pil)).long()

        ffa_diff = torch.abs(ffa_av - ffa_a)

        if self.augment:
            if random.random() > 0.5:
                cfp=TF.hflip(cfp); ffa_a=TF.hflip(ffa_a); ffa_diff=TF.hflip(ffa_diff); label=TF.hflip(label)
            if random.random() > 0.5:
                cfp=TF.vflip(cfp); ffa_a=TF.vflip(ffa_a); ffa_diff=TF.vflip(ffa_diff); label=TF.vflip(label)
            if random.random() > 0.5:
                angle = random.uniform(-15, 15)
                cfp=TF.rotate(cfp,angle); ffa_a=TF.rotate(ffa_a,angle); ffa_diff=TF.rotate(ffa_diff,angle)
                label=TF.rotate(label.unsqueeze(0), angle,
                       interpolation=TF.InterpolationMode.NEAREST).squeeze(0)
            if random.random() > 0.5:
                cfp = T.ColorJitter(brightness=0.2, contrast=0.2)(cfp)

        mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
        cfp = (cfp - mean) / std

        x = torch.cat([cfp, ffa_diff, ffa_a], dim=0)
        return {"input": x, "label": label, "case": case}


print("Dataset test করছি...")
train_idx = list(range(40))
val_idx   = list(range(40, 50))
train_ds = GAVE2Dataset(train_idx, augment=True)
val_ds   = GAVE2Dataset(val_idx,   augment=False)

sample = train_ds[0]
print(f"✅ input shape: {sample['input'].shape}")
print(f"✅ label shape: {sample['label'].shape}, unique={sample['label'].unique().tolist()}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
print(f"   Train: {len(train_ds)} | Val: {len(val_ds)}")
print("\n🎯 Cell 5 সম্পন্ন। Cell 6 run করো।")

Dataset test করছি...
✅ input shape: torch.Size([5, 1024, 1024])
✅ label shape: torch.Size([1024, 1024]), unique=[0, 1, 2, 3]
   Train: 40 | Val: 10

🎯 Cell 5 সম্পন্ন। Cell 6 run করো।


In [ ]:
# ============================================================
# CELL 6: LOSS FUNCTIONS (আগের EfficientNet/HRNet-এর হুবহু কপি)
# Focal + Tversky + clDice + Boundary
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        return ((1-pt)**self.gamma * ce).mean()


class TverskyLoss(nn.Module):
    def __init__(self, num_classes=4, alpha=0.3, beta=0.7, smooth=1.0):
        super().__init__()
        self.num_classes=num_classes; self.alpha=alpha; self.beta=beta; self.smooth=smooth
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        t_oh = F.one_hot(targets, self.num_classes).permute(0,3,1,2).float()
        dims = (0,2,3)
        TP = (probs*t_oh).sum(dims)
        FP = (probs*(1-t_oh)).sum(dims)
        FN = ((1-probs)*t_oh).sum(dims)
        tversky = (TP+self.smooth)/(TP+self.alpha*FP+self.beta*FN+self.smooth)
        return 1.0 - tversky[1:].mean()


def soft_skeletonize(mask, iters=5):
    eroded = mask.clone()
    for _ in range(iters):
        eroded = -F.max_pool2d(-eroded, 3, 1, 1)
    return F.relu(mask - eroded)


class clDiceLoss(nn.Module):
    def __init__(self, smooth=1.0, iters=5):
        super().__init__()
        self.smooth=smooth; self.iters=iters
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        total = 0.0
        for cls in [1, 2]:
            p = probs[:, cls:cls+1]
            t = (targets==cls).float().unsqueeze(1)
            sp = soft_skeletonize(p, self.iters)
            st = soft_skeletonize(t, self.iters)
            tprec = ((sp*t).sum()+self.smooth)/(sp.sum()+self.smooth)
            tsens = ((st*p).sum()+self.smooth)/(st.sum()+self.smooth)
            total += 1.0 - 2.0*tprec*tsens/(tprec+tsens+1e-8)
        return total/2.0


class BoundaryLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def _boundary(self, mask):
        eroded = -F.max_pool2d(-mask, kernel_size=3, stride=1, padding=1)
        return F.relu(mask - eroded)
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        total = 0.0
        for cls in [1, 2]:
            p = probs[:, cls:cls+1]
            t = (targets==cls).float().unsqueeze(1)
            pb = self._boundary(p); tb = self._boundary(t)
            inter = (pb*tb).sum(); union = pb.sum() + tb.sum()
            total += 1.0 - (2*inter + self.smooth)/(union + self.smooth)
        return total / 2.0


class CombinedLoss(nn.Module):
    def __init__(self, class_weights):
        super().__init__()
        self.focal    = FocalLoss(gamma=2.0, weight=class_weights)
        self.tversky  = TverskyLoss(num_classes=NUM_CLASSES, alpha=0.3, beta=0.7)
        self.cldice   = clDiceLoss(iters=5)
        self.boundary = BoundaryLoss()
    def forward(self, logits, targets):
        lf = self.focal(logits, targets)
        lt = self.tversky(logits, targets)
        lc = self.cldice(logits, targets)
        lb = self.boundary(logits, targets)
        total = 0.2*lf + 0.2*lt + 0.4*lc + 0.2*lb
        return total, lf, lt, lc


CLASS_WEIGHTS = torch.tensor([0.3, 4.0, 3.0, 6.0]).to(DEVICE)
criterion = CombinedLoss(CLASS_WEIGHTS).to(DEVICE)

with torch.no_grad():
    dl = torch.randn(2, NUM_CLASSES, 64, 64).to(DEVICE)
    dt = torch.randint(0, NUM_CLASSES, (2,64,64)).to(DEVICE)
    tot, lf, lt, lc = criterion(dl, dt)
print(f"✅ Loss test: total={tot.item():.4f} focal={lf.item():.4f} "
      f"tversky={lt.item():.4f} clDice={lc.item():.4f}")
print("🎯 Cell 6 সম্পন্ন। Cell 7 run করো।")

✅ Loss test: total=1.7163 focal=5.5450 tversky=0.7527 clDice=0.7519
🎯 Cell 6 সম্পন্ন। Cell 7 run করো।


In [ ]:
# ============================================================
# CELL 7: build_model() (HRVRL/PGNet, checkpoint+surgery একসাথে) + Train/Val/TTA
# ============================================================

def build_model():
    """
    HRVRL (PGNet architecture) — pretrained encoder+decoder load করে
    5-channel input, 4-class output-এ adapt করে।
    """
    # ধাপ ১: original architecture (3ch/4class placeholder) বানিয়ে pretrained load
    model = PGNet(input_ch=3, num_classes=4)
    model_state = model.state_dict()
    matched = {k: v for k, v in raw_state.items()
               if k in model_state and model_state[k].shape == v.shape}
    model_state.update(matched)
    model.load_state_dict(model_state)

    # ধাপ ২: stem conv 3→5 channel প্রসারণ (প্রথম ৩ pretrained, শেষ ২ নতুন)
    old_conv = model.sn_unet[0][0]
    old_weight = old_conv.weight.data
    new_conv = nn.Conv2d(5, 96, kernel_size=4, stride=4, bias=(old_conv.bias is not None))
    with torch.no_grad():
        new_conv.weight[:, :3, :, :] = old_weight
        nn.init.kaiming_normal_(new_conv.weight[:, 3:, :, :])
        if old_conv.bias is not None:
            new_conv.bias[:] = old_conv.bias.data
    model.sn_unet[0][0] = new_conv

    return model


def train_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        x      = batch['input'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(x)
            loss, _, _, _ = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_art, all_vein = [], []
    for batch in loader:
        x      = batch['input'].to(device)
        labels = batch['label'].to(device)
        logits = model(x)
        loss, _, _, _ = criterion(logits, labels)
        total_loss += loss.item()
        s = compute_dice_scores(logits, labels)
        all_art.append(s['artery_dice']); all_vein.append(s['vein_dice'])
    art  = sum(all_art)/len(all_art)
    vein = sum(all_vein)/len(all_vein)
    return total_loss/len(loader), art, vein, (art+vein)/2


@torch.no_grad()
def tta_validate(model, loader, device):
    model.eval()
    all_art, all_vein = [], []
    for batch in loader:
        x      = batch['input'].to(device)
        labels = batch['label'].to(device)
        preds_sum = torch.zeros(x.shape[0], NUM_CLASSES, IMG_SIZE, IMG_SIZE)
        for hf, vf in [(False,False),(True,False),(False,True),(True,True)]:
            xi = x.clone()
            if hf: xi = torch.flip(xi, [3])
            if vf: xi = torch.flip(xi, [2])
            lg = model(xi)
            if hf: lg = torch.flip(lg, [3])
            if vf: lg = torch.flip(lg, [2])
            preds_sum += F.softmax(lg, dim=1).cpu()
        s = compute_dice_scores(preds_sum.to(device), labels)
        all_art.append(s['artery_dice']); all_vein.append(s['vein_dice'])
    art  = sum(all_art)/len(all_art)
    vein = sum(all_vein)/len(all_vein)
    return art, vein, (art+vein)/2


# --- compute_dice_scores (Dice metric, আগের হুবহু) ---
def compute_dice_scores(logits, targets, num_classes=NUM_CLASSES, smooth=1e-6):
    preds = logits.argmax(dim=1)
    scores = {}; vessel = []
    for cls, name in [(1,'artery'), (2,'vein'), (3,'overlap')]:
        pm = (preds==cls).float(); tm = (targets==cls).float()
        inter = (pm*tm).sum(); union = pm.sum() + tm.sum()
        dice = (2*inter + smooth)/(union + smooth)
        scores[f'{name}_dice'] = dice.item()
        if cls in [1, 2]:
            vessel.append(dice.item())
    scores['mean_dice'] = sum(vessel)/len(vessel)
    return scores


# --- Smoke test ---
print("Smoke test করছি (build_model() দিয়ে fresh model)...")
model = build_model().to(DEVICE)
dummy_ds = GAVE2Dataset([0,1], augment=False)
dummy_loader = DataLoader(dummy_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
opt_test = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
scaler_test = torch.amp.GradScaler('cuda')

torch.cuda.reset_peak_memory_stats()
tr = train_epoch(model, dummy_loader, criterion, opt_test, DEVICE, scaler_test)
vl, art, vein, mean = validate_epoch(model, dummy_loader, criterion, DEVICE)
peak = torch.cuda.max_memory_allocated()/1e9
print(f"✅ train: {tr:.4f} | val: {vl:.4f} | Mean Dice: {mean:.4f} | VRAM(train+backward): {peak:.1f} GB")
print("\n🎯 Cell 7 সম্পন্ন। Cell 8 run করো।")

Smoke test করছি (build_model() দিয়ে fresh model)...
✅ train: 0.8425 | val: 0.8127 | Mean Dice: 0.0555 | VRAM(train+backward): 7.1 GB

🎯 Cell 7 সম্পন্ন। Cell 8 run করো।


In [ ]:
# ============================================================
# CLEANUP: Cell 7-এর smoke-test model/optimizer মুছে VRAM ফাঁকা করা
# Cell 8 চালানোর আগে এটা রান করো
# ============================================================

import gc

# Cell 7-এ তৈরি হওয়া variable গুলো মুছি (থাকলে)
for var_name in ['model', 'opt_test', 'scaler_test', 'dummy_loader', 'dummy_ds']:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"✅ Cleanup সম্পন্ন")
print(f"   বর্তমান allocated VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"   বর্তমান reserved VRAM:  {torch.cuda.memory_reserved()/1e9:.2f} GB")
print("\n🎯 এবার Cell 8 (run_training) নিরাপদে চালাতে পারো।")

✅ Cleanup সম্পন্ন
   বর্তমান allocated VRAM: 2.67 GB
   বর্তমান reserved VRAM:  2.68 GB

🎯 এবার Cell 8 (run_training) নিরাপদে চালাতে পারো।


In [ ]:
# ============================================================
# CELL 8: run_training() — HRVRL (PGNet), clDice loss
# ============================================================

def run_training(num_epochs=70, resume_path=None):
    train_ds = GAVE2Dataset(list(range(40)),     augment=True)
    val_ds   = GAVE2Dataset(list(range(40, 50)), augment=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    net = build_model().to(DEVICE)
    if resume_path and os.path.exists(resume_path):
        net.load_state_dict(torch.load(resume_path, map_location=DEVICE, weights_only=False))
        print(f"✅ Resumed: {resume_path}")

    criterion = CombinedLoss(CLASS_WEIGHTS).to(DEVICE)
    optimizer = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda')

    best_dice, best_epoch = 0.0, 0
    print(f"\nTraining: {num_epochs} epochs, batch={BATCH_SIZE}, res={IMG_SIZE}")
    print(f"Model: HRVRL (PGNet, ConvNeXt-Tiny encoder+decoder, pretrained) | Loss: Focal+Tversky+clDice+Boundary")
    print(f"Save: {SAVE_PATH}")
    print(f"\n{'Ep':>4} | {'TrLoss':>7} | {'ArtDice':>8} | {'VnDice':>7} | {'MnDice':>7} | {'LR':>9}")
    print("-"*58)

    for epoch in range(1, num_epochs + 1):
        tr_loss = train_epoch(net, train_loader, criterion, optimizer, DEVICE, scaler)
        do_val = (epoch % 3 == 0) or (epoch > 30) or (epoch == 1)
        if do_val:
            _, art, vein, mean = validate_epoch(net, val_loader, criterion, DEVICE)
        scheduler.step()
        torch.cuda.empty_cache()
        lr = scheduler.get_last_lr()[0]

        if do_val:
            is_best = mean > best_dice
            if is_best:
                best_dice, best_epoch = mean, epoch
                torch.save(net.state_dict(), SAVE_PATH)
            if epoch % 5 == 0 or epoch == 1 or is_best:
                marker = " ← best" if is_best else ""
                print(f"{epoch:>4} | {tr_loss:>7.4f} | {art:>8.4f} | "
                      f"{vein:>7.4f} | {mean:>7.4f} | {lr:>9.6f}{marker}")

    print(f"\n{'='*58}")
    print(f"শেষ! Best val Dice: {best_dice:.4f} (epoch {best_epoch})")
    net.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False))
    ta, tv, tm = tta_validate(net, val_loader, DEVICE)
    print(f"TTA → Artery: {ta:.4f} | Vein: {tv:.4f} | Mean: {tm:.4f}")
    print(f"\n--- তুলনা (সব একই clDice-loss, ন্যায্য) ---")
    print(f"  EfficientNet-B5 (ImageNet):       Mean Dice 0.7606")
    print(f"  HRNet-W32 (ImageNet):             Mean Dice 0.7727")
    print(f"  ConvNeXt-DINOv3 (self-superv.):   Mean Dice 0.7541")
    print(f"  HRVRL/PGNet (retinal-AV-pretrain):Mean Dice {tm:.4f}")
    print(f"{'='*58}")
    return net, tm

trained_hrvrl, final_dice = run_training(num_epochs=70)


Training: 70 epochs, batch=2, res=1024
Model: HRVRL (PGNet, ConvNeXt-Tiny encoder+decoder, pretrained) | Loss: Focal+Tversky+clDice+Boundary
Save: /content/drive/MyDrive/GAVE2_preliminary/best_model_hrvrl_pgnet.pth

  Ep |  TrLoss |  ArtDice |  VnDice |  MnDice |        LR
----------------------------------------------------------
   1 |  0.6415 |   0.6658 |  0.7350 |  0.7004 |  0.000300 ← best
   3 |  0.4036 |   0.7155 |  0.7646 |  0.7401 |  0.000299 ← best
   6 |  0.3794 |   0.7184 |  0.7762 |  0.7473 |  0.000295 ← best
   9 |  0.3678 |   0.7422 |  0.7822 |  0.7622 |  0.000288 ← best
  15 |  0.3507 |   0.7370 |  0.7803 |  0.7586 |  0.000267
  24 |  0.3097 |   0.7427 |  0.7891 |  0.7659 |  0.000221 ← best
  27 |  0.2974 |   0.7431 |  0.7899 |  0.7665 |  0.000203 ← best
  30 |  0.2969 |   0.7395 |  0.7853 |  0.7624 |  0.000184
  31 |  0.3009 |   0.7458 |  0.7884 |  0.7671 |  0.000177 ← best
  35 |  0.2946 |   0.7396 |  0.7859 |  0.7627 |  0.000150
  40 |  0.2878 |   0.7399 |  0.7887 |

# **RR Chain Reproduction — RITE → BCE → clDice → Skeleton-Recall → PGNet**



```
RITE pretrained  →[Cell 3: BCE]→        rr_second_u_ft.pth        (0.373)
                 →[Cell 4: clDice]→     rr_second_u_cldice.pth    (0.537)
                 →[Cell 5: SkelRecall]→ rr_second_u_skelrecall.pth(0.741 @n=50)
                 →[Cell 7: PGNet cache]→ rr_second_u_pgnet.pth    (0.802 @n=50)```



**CELL 1 — Setup: EffNet base + RRWNet + metric + RITE download**



```
best_model_cnn.pth (EffNet-B5 base)
training/ folder (images/, FFA_A/, FFA_AV/, av/)
Internet (downloads RITE weight automatically)```



In [ ]:
# ============================================================
# CELL 1 — setup: EffNet base, RRWNet, metric, helpers, RITE weight
# GPU ~4GB
# ============================================================
from google.colab import drive; drive.mount('/content/drive')
!pip install -q monai scikit-image scipy

import os, random, copy, gc
import numpy as np, cv2, torch
from torch import nn
import torch.nn.functional as F
from PIL import Image
from skimage import morphology, graph
from monai.networks.nets import FlexibleUNet

DEVICE="cuda"; DATA_ROOT="/content/drive/MyDrive/GAVE2_preliminary"
TRAIN_DIR=f"{DATA_ROOT}/training"; AV_DIR=f"{TRAIN_DIR}/av"
CACHE_DIR=f"{DATA_ROOT}/rr_cache"; os.makedirs(CACHE_DIR,exist_ok=True)
IMG=1024; NC=4; OW,OH=1536,1024
train_cases=[f"g_{i:03d}" for i in range(1,41)]
val_cases  =[f"g_{i:03d}" for i in range(41,51)]

# ---------- RRWNet ----------
class ConvBlock(nn.Module):
    def __init__(s,i=3,o=64,act=nn.ReLU,bias=True):
        super().__init__(); s.conv_block=nn.Sequential(nn.Conv2d(i,o,3,1,1,bias=bias),act(inplace=True),nn.Conv2d(o,o,3,1,1,bias=bias),act(inplace=True))
    def forward(s,x): return s.conv_block(x)
class UpConv(nn.Module):
    def __init__(s,i=64,o=32,bias=True): super().__init__(); s.conv=nn.ConvTranspose2d(i,o,2,2,bias=bias)
    def forward(s,x): return s.conv(x)
class UNetModule(nn.Module):
    def __init__(s,ic,oc,bc):
        super().__init__()
        s.conv1=ConvBlock(ic,bc); s.conv2=ConvBlock(bc,2*bc); s.conv3=ConvBlock(2*bc,4*bc)
        s.conv4=ConvBlock(4*bc,8*bc); s.conv5=ConvBlock(8*bc,16*bc)
        s.upconv1=UpConv(16*bc,8*bc); s.conv6=ConvBlock(16*bc,8*bc)
        s.upconv2=UpConv(8*bc,4*bc);  s.conv7=ConvBlock(8*bc,4*bc)
        s.upconv3=UpConv(4*bc,2*bc);  s.conv8=ConvBlock(4*bc,2*bc)
        s.upconv4=UpConv(2*bc,bc);    s.conv9=ConvBlock(2*bc,bc); s.outconv=nn.Conv2d(bc,oc,1,bias=True)
    def forward(s,x):
        x1=s.conv1(x); x=F.max_pool2d(x1,2,2); x2=s.conv2(x); x=F.max_pool2d(x2,2,2)
        x3=s.conv3(x); x=F.max_pool2d(x3,2,2); x4=s.conv4(x); x=F.max_pool2d(x4,2,2); x=s.conv5(x)
        x=s.upconv1(x); x=s.conv6(torch.cat((x4,x),1)); x=s.upconv2(x); x=s.conv7(torch.cat((x3,x),1))
        x=s.upconv3(x); x=s.conv8(torch.cat((x2,x),1)); x=s.upconv4(x); x=s.conv9(torch.cat((x1,x),1))
        return s.outconv(x)
class RRWNet(nn.Module):
    def __init__(s,input_ch=3,output_ch=3,base_ch=64,iterations=5):
        super().__init__(); s.first_u=UNetModule(input_ch,output_ch,base_ch); s.second_u=UNetModule(output_ch,2,base_ch); s.iterations=iterations
    def refine(s,x):
        preds=[]; bv=x[:,2:3]; p2=s.second_u(x); preds.append(torch.cat((torch.sigmoid(p2),bv),1))
        for _ in range(s.iterations):
            p2=torch.sigmoid(p2); p2=torch.cat((p2,bv),1); p2=s.second_u(p2); preds.append(torch.cat((torch.sigmoid(p2),bv),1))
        return preds

# ---------- metric (validated ruler) ----------
def topo_metric(gt,pred,thr,n):
    res=[]; pbw=(pred>thr).astype(int); pcc=morphology.label(pbw)
    gc=morphology.skeletonize(gt>0.5); gcc=morphology.label(gc); pc=morphology.skeletonize(pbw)
    gco=np.ones(gc.shape);gco[gc==0]=10000; pco=np.ones(pc.shape);pco[pc==0]=10000
    for _ in range(n):
        R,C=np.where(gc==1)
        if len(R)==0: continue
        i1=random.randint(0,len(R)-1); lbl=gcc[R[i1],C[i1]]; p1=(R[i1],C[i1])
        Rl,Cl=np.where(gcc==lbl); i2=random.randint(0,len(Rl)-1); p2=(Rl[i2],Cl[i2])
        if (pcc[p1]!=pcc[p2]) or pcc[p1]==0: res.append(0)
        else:
            Rp,Cp=np.where(pc==1); pos=np.transpose(np.asarray([Rp,Cp]))
            a=pos[np.argmin(np.sum((pos-np.asarray(p1))**2,1))]; b=pos[np.argmin(np.sum((pos-np.asarray(p2))**2,1))]
            gp,_=graph.route_through_array(gco,p1,p2); pp,_=graph.route_through_array(pco,tuple(a),tuple(b))
            gp=np.asarray(gp); pp=np.asarray(pp)
            Lg=np.sum(np.sqrt(np.sum(np.diff(gp,axis=0)**2,1))); Lp=np.sum(np.sqrt(np.sum(np.diff(pp,axis=0)**2,1)))
            if pp.shape[0]<2: res.append(2)
            elif (Lg/Lp<0.9) or (Lg/Lp>1.1): res.append(1)
            else: res.append(2)
    return res.count(0),res.count(1),res.count(2)      # INF, wrong-length, COR
def eval_topo(gt,pred,n,seed=0):
    random.seed(seed); o={}
    for ci,cn in [(0,'A'),(1,'V')]:
        inf,wl,cor=topo_metric(gt[:,:,ci],pred[:,:,ci],0.5,n); o[cn]={'COR':cor/n,'INF':inf/n}
    return o
def dsc_np(pb,gb):
    p,g=pb.astype(bool),gb.astype(bool); d=p.sum()+g.sum()
    return 1.0 if d==0 else 2*(p&g).sum()/d
def to_native(a): return cv2.resize(a.astype(np.float32),(OW,OH),interpolation=cv2.INTER_LINEAR)
def parse_av(p,res=None):
    av=np.array(Image.open(p).convert("RGB"))
    if res: av=cv2.resize(av,(res,res),interpolation=cv2.INTER_NEAREST)
    R,G,B=av[:,:,0],av[:,:,1],av[:,:,2]
    art=(R>150)&(G<50)&(B<50); vein=(R<50)&(G<50)&(B>150); ov=(R<50)&(G>150)&(B<50)
    return np.stack([art|ov,vein|ov,art|vein|ov],-1).astype(np.float32)

# ---------- input helpers ----------
_M=torch.tensor([0.485,0.456,0.406]).view(3,1,1); _S=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
def _rgb(p):
    im=Image.open(p).convert("RGB").resize((IMG,IMG),Image.BILINEAR); return torch.from_numpy(np.array(im,np.float32)/255.).permute(2,0,1)
def _gray(p):
    im=Image.open(p).convert("L").resize((IMG,IMG),Image.BILINEAR); return torch.from_numpy(np.array(im,np.float32)/255.).unsqueeze(0)
def build_input(c,d):
    cfp=_rgb(f"{d}/images/{c}.png"); fa=_gray(f"{d}/FFA_A/{c}.png"); fav=_gray(f"{d}/FFA_AV/{c}.png")
    return torch.cat([(cfp-_M)/_S, torch.abs(fav-fa), fa],0)
@torch.no_grad()
def tta(model,x):
    x=x.unsqueeze(0).to(DEVICE); s=torch.zeros(1,NC,x.shape[-2],x.shape[-1])
    for hf,vf in [(0,0),(1,0),(0,1),(1,1)]:
        xi=x.clone()
        if hf: xi=torch.flip(xi,[3])
        if vf: xi=torch.flip(xi,[2])
        with torch.amp.autocast('cuda'): lg=model(xi)
        if hf: lg=torch.flip(lg,[3])
        if vf: lg=torch.flip(lg,[2])
        s+=F.softmax(lg.float(),1).cpu()
    return (s/4).squeeze(0).numpy()
def to_avv(p):
    bg,art,vein,ov=p; return np.stack([art+ov,vein+ov,art+vein+ov],-1).astype(np.float32)

# ---------- EffNet base ----------
base=FlexibleUNet(in_channels=5,out_channels=4,backbone="efficientnet-b5",pretrained=False,spatial_dims=2)
base.load_state_dict(torch.load(f"{DATA_ROOT}/best_model_cnn.pth",map_location='cpu',weights_only=False),strict=True)
base=base.to(DEVICE).eval()
print("✓ EffNet base (0.76) loaded")

# ---------- RITE refinement weight ----------
RITE_W=f"{CACHE_DIR}/rrwnet_RITE_refinement.pth"
if not os.path.exists(RITE_W):
    os.system(f"wget -q -O {RITE_W} https://github.com/j-morano/rrwnet/releases/download/weights/rrwnet_RITE_refinement.pth")
ok=os.path.exists(RITE_W) and os.path.getsize(RITE_W)>1_000_000
print(f"✓ RITE weight: {'downloaded' if ok else 'FAIL'} ({os.path.getsize(RITE_W)//1024 if ok else 0} KB)")

_rr=RRWNet(iterations=5)
_sd=torch.load(RITE_W,map_location='cpu',weights_only=False)
if isinstance(_sd,dict) and 'state_dict' in _sd: _sd=_sd['state_dict']
_rr.load_state_dict(_sd,strict=False)
print(f"✓ RRWNet loads from RITE — missing keys: {len(set(_rr.state_dict())-set(_sd))}")
print("\n✓ Cell 1 done")

**CELL 2 — EffNet base prediction cache**



```
Required: Cell 1 run. Files: best_model_cnn.pth (loaded in Cell 1), training/.
Produces: rr_cache/cache.npz
Expect: DSC=0.7599 ... TopoScore=0.249


```



In [ ]:
# ============================================================
# CELL 2 — EffNet base prediction cache (train 40 + val 10)
# ============================================================
print("Train cache (40 case)...")
Xtr=[]; Ytr=[]
for c in train_cases:
    Xtr.append(to_avv(tta(base,build_input(c,TRAIN_DIR))).astype(np.float16))
    Ytr.append(parse_av(f"{AV_DIR}/{c}.png",IMG).astype(np.float16))
    print('.',end='',flush=True)
Xtr=np.stack(Xtr).astype(np.float32); Ytr=np.stack(Ytr).astype(np.float32)
print(f"\n  Xtr{Xtr.shape} Ytr{Ytr.shape}")

print("Val cache (10 case)...")
Xval=[]
for c in val_cases:
    Xval.append(to_avv(tta(base,build_input(c,TRAIN_DIR))).astype(np.float16)); print('.',end='',flush=True)
Xval=np.stack(Xval).astype(np.float32)
GTval_native=[parse_av(f"{AV_DIR}/{c}.png") for c in val_cases]
print(f"\n  Xval{Xval.shape}")

np.savez_compressed(f"{CACHE_DIR}/cache.npz",Xtr=Xtr.astype(np.float16),Ytr=Ytr.astype(np.float16),
                    Xval=Xval.astype(np.float16),GTval=np.stack(GTval_native),val_cases=np.array(val_cases))
print(f"✓ saved: {CACHE_DIR}/cache.npz ({os.path.getsize(CACHE_DIR+'/cache.npz')//10**6} MB)")

# base raw baseline
TS,COR,INF,D=[],[],[],[]
for i in range(len(val_cases)):
    pn=to_native(Xval[i]); gt=GTval_native[i]; t=eval_topo(gt,pn,35,0)
    sA=0.5*t['A']['COR']+0.5*(1-t['A']['INF']); sV=0.5*t['V']['COR']+0.5*(1-t['V']['INF'])
    TS.append((sA+sV)/2); COR.append((t['A']['COR']+t['V']['COR'])/2)
    INF.append((t['A']['INF']+t['V']['INF'])/2)
    D.append((dsc_np(pn[:,:,0]>0.5,gt[:,:,0])+dsc_np(pn[:,:,1]>0.5,gt[:,:,1]))/2)
print(f"BASE raw (cache): DSC={np.mean(D):.4f} COR={np.mean(COR):.3f} INF={np.mean(INF):.3f} TopoScore={np.mean(TS):.3f}")
print("→ DSC~0.76, TopoScore~0.25 হলে cache সঠিক ✓")

**CELL 3 — BCE fine-tune (RITE → GAVE2) → rr_second_u_ft.pth**



```
Required: Cells 1–2. Files: rr_cache/cache.npz, rr_cache/rrwnet_RITE_refinement.pth
Expect: TopoScore ~0.37. First 2 epochs may dip (RITE prior adjusting) — normal.
```



In [ ]:
# ============================================================
# CELL 3 — RR second_u BCE fine-tune (RITE→GAVE2)
#   base frozen (cache), res 768, iter 8 | target 0.249 → ~0.373
# GPU peak ~4-5GB
# ============================================================
_z=np.load(f"{CACHE_DIR}/cache.npz",allow_pickle=True)
Xtr=_z['Xtr'].astype(np.float32); Ytr=_z['Ytr'].astype(np.float32)
Xval=_z['Xval'].astype(np.float32); GTval_native=[g.astype(np.float32) for g in _z['GTval']]
val_cases=list(_z['val_cases']); print(f"Xtr{Xtr.shape} Xval{Xval.shape}")

TRAIN_RES=768; ITERS=8; EPOCHS=20; LR=1e-4

def eval_rr(net,n=35):
    net.eval(); TS,COR,INF,D=[],[],[],[]
    for i in range(len(val_cases)):
        with torch.no_grad():
            x=torch.from_numpy(np.ascontiguousarray(Xval[i])).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
            r=net.refine(x)[-1].squeeze(0).cpu().numpy().transpose(1,2,0); torch.cuda.empty_cache()
        pn=to_native(r); gt=GTval_native[i]; t=eval_topo(gt,pn,n,0)
        sA=0.5*t['A']['COR']+0.5*(1-t['A']['INF']); sV=0.5*t['V']['COR']+0.5*(1-t['V']['INF'])
        TS.append((sA+sV)/2); COR.append((t['A']['COR']+t['V']['COR'])/2)
        INF.append((t['A']['INF']+t['V']['INF'])/2)
        D.append((dsc_np(pn[:,:,0]>0.5,gt[:,:,0])+dsc_np(pn[:,:,1]>0.5,gt[:,:,1]))/2)
    return np.mean(D),np.mean(COR),np.mean(INF),np.mean(TS)

rr_ft=RRWNet(iterations=ITERS)
_sd=torch.load(RITE_W,map_location='cpu',weights_only=False)
if isinstance(_sd,dict) and 'state_dict' in _sd: _sd=_sd['state_dict']
rr_ft.load_state_dict(_sd,strict=False)                  # RITE warm-start
rr_ft=rr_ft.to(DEVICE)
for p in rr_ft.first_u.parameters():  p.requires_grad_(False)
for p in rr_ft.second_u.parameters(): p.requires_grad_(True)
opt=torch.optim.AdamW(rr_ft.second_u.parameters(),lr=LR,weight_decay=1e-5)
bce=nn.BCELoss()

def refine_train(x):
    bv=x[:,2:3]; outs=[]; p2=rr_ft.second_u(x); outs.append(torch.sigmoid(p2))
    for _ in range(ITERS):
        p=torch.cat((torch.sigmoid(p2),bv),1); p2=rr_ft.second_u(p); outs.append(torch.sigmoid(p2))
    return outs
def rz(a,r): return cv2.resize(a.astype(np.float32),(r,r),interpolation=cv2.INTER_LINEAR)

best_ts,best_state=-1.0,None
print(f"Target: 0.249 → ~0.373 | BCE, iter{ITERS}, res{TRAIN_RES}\n")
for ep in range(1,EPOCHS+1):
    rr_ft.second_u.train(); order=np.random.permutation(len(Xtr)); tot=0.0
    for idx in order:
        x=torch.from_numpy(rz(Xtr[idx],TRAIN_RES)).permute(2,0,1).unsqueeze(0).to(DEVICE)
        yb=rz(Ytr[idx],TRAIN_RES)
        yA=torch.from_numpy(yb[:,:,0]).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        yV=torch.from_numpy(yb[:,:,1]).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        outs=refine_train(x); loss=0.0
        for o in outs:
            oA=o[:,0:1].clamp(1e-6,1-1e-6); oV=o[:,1:2].clamp(1e-6,1-1e-6)
            loss=loss+bce(oA,yA)+bce(oV,yV)
        loss=loss/len(outs); opt.zero_grad(); loss.backward(); opt.step()
        tot+=loss.item(); del outs,loss,x; torch.cuda.empty_cache()
    if ep%2==0 or ep==1 or ep==EPOCHS:
        d,c_,i_,ts=eval_rr(rr_ft,35); flag=""
        if ts>best_ts and d>=0.68:
            best_ts=ts; best_state=copy.deepcopy(rr_ft.second_u.state_dict()); flag="  ✓ saved"
        print(f"ep{ep:02d} loss={tot/len(Xtr):.4f} | DSC={d:.3f} COR={c_:.3f} INF={i_:.3f} TopoScore={ts:.3f}{flag}")
    else:
        print(f"ep{ep:02d} loss={tot/len(Xtr):.4f} | (eval skip)")

if best_state is not None:
    torch.save(best_state,f"{CACHE_DIR}/rr_second_u_ft.pth")
    print(f"\n✓ BCE best saved: TopoScore={best_ts:.3f} → rr_cache/rr_second_u_ft.pth")
else:
    print("\n✗ improve হয়নি")

**CELL 4 — clDice → rr_second_u_cldice.pth**


```
Required: Cells 1–3. Files: rr_cache/cache.npz, rr_cache/rr_second_u_ft.pth
Expect: TopoScore ~0.537
```



In [ ]:
# ============================================================
# CELL 4 — BCE + 0.5·clDice (warm-start ft) | target ~0.537
# GPU peak ~4-5GB
# ============================================================
TRAIN_RES=768; ITERS=8; EPOCHS=20; LR=8e-5; LAMBDA=0.5; SKEL_ITERS=5

def s_erode(x): return -F.max_pool2d(-x,(3,3),1,1)
def s_open(x):  return F.max_pool2d(s_erode(x),(3,3),1,1)
def s_skel(x,it=SKEL_ITERS):
    x1=s_open(x); sk=F.relu(x-x1)
    for _ in range(it):
        x=s_erode(x); x1=s_open(x); d=F.relu(x-x1); sk=sk+F.relu(d-sk*d)
    return sk
def cldice(pred,tgt,eps=1e-5):
    sp=s_skel(pred); st=s_skel(tgt)
    tp=(torch.sum(sp*tgt)+eps)/(torch.sum(sp)+eps); ts=(torch.sum(st*pred)+eps)/(torch.sum(st)+eps)
    return 1.0-2.0*tp*ts/(tp+ts)

rr_ft=RRWNet(iterations=ITERS)
rr_ft.second_u.load_state_dict(
    torch.load(f"{CACHE_DIR}/rr_second_u_ft.pth",map_location='cpu',weights_only=False),strict=True)
rr_ft=rr_ft.to(DEVICE)
for p in rr_ft.first_u.parameters():  p.requires_grad_(False)
for p in rr_ft.second_u.parameters(): p.requires_grad_(True)
opt=torch.optim.AdamW(rr_ft.second_u.parameters(),lr=LR,weight_decay=1e-5)
bce=nn.BCELoss()

def refine_train(x):
    bv=x[:,2:3]; outs=[]; p2=rr_ft.second_u(x); outs.append(torch.sigmoid(p2))
    for _ in range(ITERS):
        p=torch.cat((torch.sigmoid(p2),bv),1); p2=rr_ft.second_u(p); outs.append(torch.sigmoid(p2))
    return outs

best_ts,best_state=-1.0,None
print(f"Target: 0.537 | BCE+{LAMBDA}·clDice, iter{ITERS}, res{TRAIN_RES}\n")
for ep in range(1,EPOCHS+1):
    rr_ft.second_u.train(); order=np.random.permutation(len(Xtr)); tot=0.0
    for idx in order:
        x=torch.from_numpy(rz(Xtr[idx],TRAIN_RES)).permute(2,0,1).unsqueeze(0).to(DEVICE)
        yb=rz(Ytr[idx],TRAIN_RES)
        yA=torch.from_numpy(yb[:,:,0]).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        yV=torch.from_numpy(yb[:,:,1]).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        outs=refine_train(x); loss=0.0
        for o in outs:
            oA=o[:,0:1].clamp(1e-6,1-1e-6); oV=o[:,1:2].clamp(1e-6,1-1e-6)
            loss=loss+bce(oA,yA)+bce(oV,yV)+LAMBDA*(cldice(oA,yA)+cldice(oV,yV))
        loss=loss/len(outs); opt.zero_grad(); loss.backward(); opt.step()
        tot+=loss.item(); del outs,loss,x; torch.cuda.empty_cache()
    if ep%3==0 or ep==1 or ep==EPOCHS:
        d,c_,i_,ts=eval_rr(rr_ft,35); flag=""
        if ts>best_ts and d>=0.68:
            best_ts=ts; best_state=copy.deepcopy(rr_ft.second_u.state_dict())
            torch.save(best_state,f"{DATA_ROOT}/rr_second_u_cldice.pth"); flag="  ✓ saved"
        print(f"ep{ep:02d} loss={tot/len(Xtr):.4f} | DSC={d:.3f} COR={c_:.3f} INF={i_:.3f} TopoScore={ts:.3f}{flag}")
    else:
        print(f"ep{ep:02d} loss={tot/len(Xtr):.4f} | (eval skip)")

if best_state is None:
    torch.save(rr_ft.second_u.state_dict(),f"{DATA_ROOT}/rr_second_u_cldice.pth")
    print("\n⚠ floor 0.68 পেরোয়নি — শেষ epoch manually saved")
else:
    print(f"\n✓ clDice best: TopoScore={best_ts:.3f} → rr_second_u_cldice.pth")

**CELL 5 — Skeleton-Recall → rr_second_u_skelrecall.pth**



```
Required: Cells 1–4. Files: rr_cache/cache.npz, rr_cache/rr_second_u_ft.pth
(Note: the original warm-started from ft.pth, NOT from cldice.)
Expect: TopoScore ~0.73 @n=35 (≈0.741 @n=50)
```




In [ ]:
# ============================================================
# CELL 5 — BCE + 0.5·Skeleton-Recall (warm-start ft) | target >0.537
# GPU peak ~5-6GB
# ============================================================
from skimage.morphology import skeletonize as skl, binary_dilation
TRAIN_RES=768; ITERS=8; EPOCHS=24; LR=8e-5; LAMBDA_SKEL=0.5; EVAL_EVERY=3

def tubed_skeleton(gt_bin, dilate=2):
    sk=skl(gt_bin>0.5)
    for _ in range(dilate): sk=binary_dilation(sk)
    return sk.astype(np.float32)
def skeleton_recall_loss(pred, skel, eps=1e-5):
    return 1.0-((pred*skel).sum()+eps)/(skel.sum()+eps)

print("GT tubed-skeleton তৈরি...")
SkA=[tubed_skeleton(Ytr[i][:,:,0],2) for i in range(len(Ytr))]
SkV=[tubed_skeleton(Ytr[i][:,:,1],2) for i in range(len(Ytr))]
print("  done")
def rz1(a,r): return cv2.resize(a.astype(np.float32),(r,r),interpolation=cv2.INTER_LINEAR).reshape(r,r)

rr_ft=RRWNet(iterations=ITERS)
rr_ft.second_u.load_state_dict(
    torch.load(f"{CACHE_DIR}/rr_second_u_ft.pth",map_location='cpu',weights_only=False),strict=True)
rr_ft=rr_ft.to(DEVICE)
for p in rr_ft.first_u.parameters():  p.requires_grad_(False)
for p in rr_ft.second_u.parameters(): p.requires_grad_(True)
opt=torch.optim.AdamW(rr_ft.second_u.parameters(),lr=LR,weight_decay=1e-5)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=1e-6)
bce=nn.BCELoss()

from torch.utils.checkpoint import checkpoint
def _su(inp): return checkpoint(rr_ft.second_u, inp, use_reentrant=False)
def refine_train(x):
    bv=x[:,2:3]; outs=[]; p2=_su(x); outs.append(torch.sigmoid(p2))
    for _ in range(ITERS):
        p=torch.cat((torch.sigmoid(p2),bv),1); p2=_su(p); outs.append(torch.sigmoid(p2))
    return outs

best_ts=0.537; best_state=None          # clDice best থেকে শুরু
print(f"\nTarget: >0.537 | BCE + {LAMBDA_SKEL}·SkeletonRecall, iter{ITERS}, res{TRAIN_RES}\n")
for ep in range(1,EPOCHS+1):
    rr_ft.second_u.train(); order=np.random.permutation(len(Xtr)); tot=0.0
    for idx in order:
        x=torch.from_numpy(rz(Xtr[idx],TRAIN_RES)).permute(2,0,1).unsqueeze(0).to(DEVICE)
        yb=rz(Ytr[idx],TRAIN_RES)
        yA=torch.from_numpy(yb[:,:,0]).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        yV=torch.from_numpy(yb[:,:,1]).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        skA=torch.from_numpy(rz1(SkA[idx],TRAIN_RES)).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        skV=torch.from_numpy(rz1(SkV[idx],TRAIN_RES)).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        outs=refine_train(x); loss=0.0
        for o in outs:
            oA=o[:,0:1].clamp(1e-6,1-1e-6); oV=o[:,1:2].clamp(1e-6,1-1e-6)
            loss=loss+bce(oA,yA)+bce(oV,yV) \
                     +LAMBDA_SKEL*(skeleton_recall_loss(oA,skA)+skeleton_recall_loss(oV,skV))
        loss=loss/len(outs); opt.zero_grad(); loss.backward(); opt.step()
        tot+=loss.item(); del outs,loss,x,yA,yV,skA,skV; torch.cuda.empty_cache()
    sched.step()
    if ep%EVAL_EVERY==0 or ep==1 or ep==EPOCHS:
        d,c_,i_,ts=eval_rr(rr_ft,35); flag=""
        if ts>best_ts and d>=0.60:              # floor 0.60 (original ran 0.68 → nothing auto-saved)
            best_ts=ts; best_state=copy.deepcopy(rr_ft.second_u.state_dict())
            torch.save(best_state,f"{DATA_ROOT}/rr_second_u_skelrecall.pth"); flag="  ✓ NEW BEST"
        print(f"ep{ep:02d} loss={tot/len(Xtr):.4f} lr={sched.get_last_lr()[0]:.1e} | DSC={d:.3f} COR={c_:.3f} INF={i_:.3f} TopoScore={ts:.3f}{flag}")
    else:
        print(f"ep{ep:02d} loss={tot/len(Xtr):.4f} | (eval skip)")

if best_state is None:                          # original fallback: manual save
    torch.save(rr_ft.second_u.state_dict(), f"{DATA_ROOT}/rr_second_u_skelrecall.pth")
    print("\n⚠ auto-save হয়নি — শেষ epoch manually saved (মূল run-এ এটাই হয়েছিল)")
else:
    print(f"\n✓ Skeleton-Recall: TopoScore={best_ts:.3f} → rr_second_u_skelrecall.pth")

**CELL 6 — PGNet base + PGNet prediction cache**


```
Required files: best_model_hrvrl_pgnet.pth, training/
Produces: rr_cache_pgnet/cache.npz
(Cell 1 must have been run for RRWNet/metric/helpers.)
Expect: best around ep03, TopoScore ~0.777 @n=35 (≈0.802 @n=50).
```



In [ ]:
# ============================================================
# CELL 7 — RR retrain on PGNet cache (warm-start skelrecall)
#   BCE + 0.5·SkelRecall | res 768, iter 8, 25 ep, LR 8e-5, floor 0.60
#   → rr_second_u_pgnet.pth   (expect ~0.777 @n=35 / 0.802 @n=50)
# GPU peak ~6GB
# ============================================================
from skimage.morphology import skeletonize as skl, binary_dilation
from torch.utils.checkpoint import checkpoint

TRAIN_RES=768; ITERS=8; EPOCHS=25; LR=8e-5; LAM=0.5; DSC_FLOOR=0.60; EVAL_EVERY=3

def tubed_sk(g,d=2):
    sk=skl(g>0.5)
    for _ in range(d): sk=binary_dilation(sk)
    return sk.astype(np.float32)
def skel_loss(pred,skel,eps=1e-5):
    return 1.0-((pred*skel).sum()+eps)/(skel.sum()+eps)
SkA=[tubed_sk(Ytr[i][:,:,0]) for i in range(len(Ytr))]
SkV=[tubed_sk(Ytr[i][:,:,1]) for i in range(len(Ytr))]

rr_ft=RRWNet(iterations=ITERS)
rr_ft.second_u.load_state_dict(
    torch.load(f"{DATA_ROOT}/rr_second_u_skelrecall.pth",map_location='cpu',weights_only=False),strict=True)
rr_ft=rr_ft.to(DEVICE)
for p in rr_ft.first_u.parameters():  p.requires_grad_(False)
for p in rr_ft.second_u.parameters(): p.requires_grad_(True)
opt=torch.optim.AdamW(rr_ft.second_u.parameters(),lr=LR,weight_decay=1e-5)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=1e-6)
bce=nn.BCELoss()

def _su(inp): return checkpoint(rr_ft.second_u,inp,use_reentrant=False)
def refine_train(x):
    bv=x[:,2:3]; outs=[]; p2=_su(x); outs.append(torch.sigmoid(p2))
    for _ in range(ITERS):
        p=torch.cat((torch.sigmoid(p2),bv),1); p2=_su(p); outs.append(torch.sigmoid(p2))
    return outs

@torch.no_grad()
def eval_rr2(n=35):
    rr_ft.eval(); TS,D=[],[]
    for i in range(len(val_cases)):
        x=torch.from_numpy(np.ascontiguousarray(Xval[i])).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
        r=rr_ft.refine(x)[-1].squeeze(0).cpu().numpy().transpose(1,2,0); torch.cuda.empty_cache()
        pn=to_native(r); gt=GTval_native[i]; t=eval_topo(gt,pn,n,0)
        sA=0.5*t['A']['COR']+0.5*(1-t['A']['INF']); sV=0.5*t['V']['COR']+0.5*(1-t['V']['INF'])
        TS.append((sA+sV)/2); D.append((dsc_np(pn[:,:,0]>0.5,gt[:,:,0])+dsc_np(pn[:,:,1]>0.5,gt[:,:,1]))/2)
    return np.mean(D),np.mean(TS)

best_ts=-1.0; best_state=None
print(f"RR retrain on PGNet | BCE+{LAM}·SkelRecall, {EPOCHS}ep, DSC≥{DSC_FLOOR}\n")
for ep in range(1,EPOCHS+1):
    rr_ft.second_u.train(); order=np.random.permutation(len(Xtr)); tot=0.0
    for idx in order:
        x=torch.from_numpy(rz(Xtr[idx],TRAIN_RES)).permute(2,0,1).unsqueeze(0).to(DEVICE)
        yb=rz(Ytr[idx],TRAIN_RES)
        yA=torch.from_numpy(yb[:,:,0]).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        yV=torch.from_numpy(yb[:,:,1]).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        skA=torch.from_numpy(rz1(SkA[idx],TRAIN_RES)).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        skV=torch.from_numpy(rz1(SkV[idx],TRAIN_RES)).view(1,1,TRAIN_RES,TRAIN_RES).to(DEVICE)
        outs=refine_train(x); loss=0.0
        for o in outs:
            oA=o[:,0:1].clamp(1e-6,1-1e-6); oV=o[:,1:2].clamp(1e-6,1-1e-6)
            loss=loss+bce(oA,yA)+bce(oV,yV)+LAM*(skel_loss(oA,skA)+skel_loss(oV,skV))
        loss=loss/len(outs); opt.zero_grad(); loss.backward(); opt.step()
        tot+=loss.item(); del outs,loss,x,yA,yV,skA,skV; torch.cuda.empty_cache()
    sched.step()
    if ep%EVAL_EVERY==0 or ep==1 or ep==EPOCHS:
        d,ts=eval_rr2(35); flag=""
        if ts>best_ts and d>=DSC_FLOOR:
            best_ts=ts; best_state=copy.deepcopy(rr_ft.second_u.state_dict())
            torch.save(best_state,f"{DATA_ROOT}/rr_second_u_pgnet.pth"); flag="  ✓ BEST"
        print(f"ep{ep:02d} loss={tot/len(Xtr):.4f} lr={sched.get_last_lr()[0]:.1e} | DSC={d:.3f} TopoScore={ts:.3f}{flag}")
    else:
        print(f"ep{ep:02d} loss={tot/len(Xtr):.4f} | (skip)")

print(f"\n✓ best TopoScore={best_ts:.3f} → rr_second_u_pgnet.pth")